In [ ]:
import csv
import json
import re
import secrets
from collections import Counter
from pathlib import Path
import spotipy
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyOAuth

## Settings

In [ ]:
PLAYLIST_NAME = 'Amazon Import'
PUBLIC = False        # script: --public
CSV_PATH = None       # script: --csv; None = grab data/data/track_list.csv

## Paths and constants

In [ ]:
# notebooks have no __file__; VS Code runs a notebook in its own folder, so that's where files go
HERE = Path.cwd()
DEFAULT_CSV = HERE.parent / 'grab data' / 'data' / 'track_list.csv'
MATCHED_CSV = HERE / 'matched.csv'
NOT_FOUND_CSV = HERE / 'not_found.csv'
CACHE_FILE = HERE / 'search_cache.json'
REDIRECT_URI = 'http://127.0.0.1:8888/callback'
SCOPE = 'playlist-modify-private playlist-modify-public'

# best to worst
MATCH_ORDER = ['exact', 'title', 'version', 'artist', 'guess']
# words that mean a different recording than the one asked for (remasters are fine)
VERSION_WORDS = ['live', 'remix', 'mix', 'acoustic', 'demo', 'edit', 'version', 'instrumental',
                 'slowed', 'sped up', 'reverb', 'karaoke', 'cover', 'session']
# amazon rounds to whole seconds, and releases of one recording can differ by a few
LENGTH_TOLERANCE_MS = 5000

CSV = Path(CSV_PATH) if CSV_PATH else DEFAULT_CSV

## Reading the track list
The pairs to search for, and the Amazon lengths to check against.

In [ ]:
def read_tracks(csv_path):
    with open(csv_path, newline='', encoding='utf-8') as file:
        return [(row['trackname'].strip(), row['artistname'].strip()) for row in csv.DictReader(file)]


def read_durations(csv_path):
    """{(trackname, artistname): amazon length in ms}, for rows that have a trackduration."""
    with open(csv_path, newline='', encoding='utf-8') as file:
        return {(row['trackname'].strip(), row['artistname'].strip()): int(row['trackduration'])
                for row in csv.DictReader(file) if (row.get('trackduration') or '').strip()}

## Cleaning up names for comparing

In [ ]:
def simplify(name):
    # drop "(feat. ...)", "[Remastered]", " - Live" etc. which often break exact searches
    name = re.sub(r'\s*[\(\[].*?[\)\]]', '', name)
    return name.split(' - ')[0].strip()


def key(name):
    # "My Way [Explicit]" -> "myway"; keeps version info like "(Slowed & Reverbed)"
    name = re.sub(r'\s*[\(\[]explicit[\)\]]', '', name, flags=re.I)
    return re.sub(r'[^a-z0-9]', '', name.lower())


def artist_key(name):
    # "The Goo Goo Dolls" and "Goo Goo Dolls" should compare equal
    k = key(simplify(name))
    return k[3:] if k.startswith('the') else k


def version_words(name):
    return {w for w in VERSION_WORDS if re.search(rf'\b{w}\b', name, flags=re.I)}

## Labelling a result by name

In [ ]:
def rank(trackname, artistname, found_name, found_artists):
    """how well a spotify result matches: 'exact', 'title', 'version', 'artist' or 'guess'."""
    # amazon puts features in the artist column ("ODECORE [feat. Odetari & 9lives]"), so compare on the main artist only
    wanted = artist_key(artistname)
    artist_hit = bool(wanted) and any(artist_key(a) and artist_key(a) in wanted for a in found_artists)
    if not artist_hit:
        return 'guess'
    if key(found_name) == key(trackname):
        return 'exact'
    if key(simplify(found_name)) == key(simplify(trackname)):
        # "Song 2 - 2012 Remaster" is fine, "Closer - Nine Inch Noize Version" is not
        return 'version' if version_words(found_name) - version_words(trackname) else 'title'
    return 'artist'

## Comparing lengths and choosing between results

In [ ]:
def length_diff(wanted_ms, found):
    """spotify length minus amazon length in ms, or None if either is unknown."""
    if wanted_ms is None or found is None or found.get('duration_ms') is None:
        return None
    return found['duration_ms'] - wanted_ms


def format_length_diff(wanted_ms, found):
    """'+2s' / '-41s', or '' if unknown."""
    diff = length_diff(wanted_ms, found)
    return '' if diff is None else f'{diff / 1000:+.0f}s'


def length_rank(wanted_ms, found):
    """0 = lengths agree, 1 = unknown, 2 = lengths differ."""
    diff = length_diff(wanted_ms, found)
    if diff is None:
        return 1
    return 0 if abs(diff) <= LENGTH_TOLERANCE_MS else 2


def preference(found, wanted_ms):
    """sort key for choosing between results, lower is better: right artist first,
    then a matching length (it tells versions apart), then the name match."""
    return (found['match'] == 'guess', length_rank(wanted_ms, found), MATCH_ORDER.index(found['match']))


def good_enough(found, wanted_ms):
    """whether to stop searching for this track."""
    if length_rank(wanted_ms, found) == 1:
        return found['match'] in ('exact', 'title')
    # with a length to compare, a matching length confirms even a "version" name
    # (e.g. "hotline (edit)" at amazon's 1:01), and a different length keeps searching
    return length_rank(wanted_ms, found) == 0 and found['match'] in ('exact', 'title', 'version')

## Searching for one track

In [ ]:
def find_track(sp, trackname, artistname, wanted_ms=None):
    """return the best result as {'name', 'artists', 'uri', 'duration_ms', 'match'}, or None."""
    artist = simplify(artistname)
    queries = [
        f'track:"{trackname}" artist:"{artist}"',
        f'{trackname} {artist}',
        f'track:"{simplify(trackname)}" artist:"{artist}"',
        f'{simplify(trackname)} {artist}',
    ]
    best = None
    for query in queries:
        results = sp.search(q=query, type='track', limit=10) or {}
        for item in results.get('tracks', {}).get('items', []):
            found = {
                'name': item['name'],
                'artists': [a['name'] for a in item['artists']],
                'uri': item['uri'],
                'duration_ms': item.get('duration_ms'),
            }
            found['match'] = rank(trackname, artistname, found['name'], found['artists'])
            if best is None or preference(found, wanted_ms) < preference(best, wanted_ms):
                best = found
        # good enough, save the remaining searches
        if best and good_enough(best, wanted_ms):
            return best
    return best

## Search cache

In [ ]:
def cache_key(trackname, artistname):
    return f'{trackname}\t{artistname}'


def load_cache():
    if CACHE_FILE.exists():
        cache = json.loads(CACHE_FILE.read_text(encoding='utf-8'))
        # results saved before lengths were compared have no duration_ms:
        # keep the exact ones, search the doubtful ones again with the length
        return {k: v for k, v in cache.items() if v is None or v['match'] == 'exact' or 'duration_ms' in v}
    # first run with the cache: reuse exact matches from an earlier matched.csv so only the doubtful tracks get searched again
    cache = {}
    if MATCHED_CSV.exists():
        with open(MATCHED_CSV, newline='', encoding='utf-8') as file:
            for row in csv.DictReader(file):
                artists = row['spotify_artists'].split(', ')
                if rank(row['trackname'], row['artistname'], row['spotify_track'], artists) == 'exact':
                    cache[cache_key(row['trackname'], row['artistname'])] = {
                        'name': row['spotify_track'], 'artists': artists, 'uri': row['uri'], 'match': 'exact'}
        print(f'reusing {len(cache)} exact matches from {MATCHED_CSV.name}')
    return cache


def save_cache(cache):
    CACHE_FILE.write_text(json.dumps(cache, indent=1, ensure_ascii=False), encoding='utf-8')

## Searching every track and writing the review files

In [ ]:
def search_all(sp, tracks, durations):
    """fill the cache for every track. Returns False if stopped early by an API error."""
    cache = load_cache()
    todo = [t for t in tracks if cache_key(*t) not in cache]
    print(f'{len(tracks) - len(todo)} tracks already matched, {len(todo)} to search')
    for n, (trackname, artistname) in enumerate(todo, 1):
        wanted_ms = durations.get((trackname, artistname))
        try:
            found = find_track(sp, trackname, artistname, wanted_ms)
        except spotipy.SpotifyException as e:
            if e.http_status == 429:
                print(f'\nrate limited by Spotify: {e.msg}')
            else:
                print(f'\nSpotify error: {e}')
            print(f'progress saved ({n - 1}/{len(todo)} searched), run again later to continue')
            return False
        cache[cache_key(trackname, artistname)] = found
        save_cache(cache)
        if found is None:
            print(f'[{n}/{len(todo)}] NOT FOUND  {trackname} - {artistname}')
        else:
            print(f'[{n}/{len(todo)}] {found["match"]:7} {format_length_diff(wanted_ms, found):>5} '
                  f'{trackname} - {artistname}  ->  {found["name"]} - {", ".join(found["artists"])}')
    return True


def write_reports(tracks, durations):
    """write matched.csv / not_found.csv from the cache. Returns the unique uris, in CSV order."""
    cache = load_cache()
    matched, missing, uris = [], [], []
    for trackname, artistname in tracks:
        found = cache[cache_key(trackname, artistname)]
        if found is None:
            missing.append((trackname, artistname))
            continue
        wanted_ms = durations.get((trackname, artistname))
        # most doubtful first: wrong artist, then a different length, then an unknown one, then the worst name match
        doubt = (found['match'] != 'guess', -length_rank(wanted_ms, found), MATCH_ORDER[::-1].index(found['match']))
        matched.append((doubt, (found['match'], format_length_diff(wanted_ms, found), trackname, artistname,
                                found['name'], ', '.join(found['artists']), found['uri'])))
        if found['uri'] not in uris:
            uris.append(found['uri'])

    matched.sort(key=lambda pair: pair[0])
    length_ranks = [-doubt[1] for doubt, _ in matched]
    matched = [row for _, row in matched]
    with open(MATCHED_CSV, 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['match', 'length_diff', 'trackname', 'artistname', 'spotify_track', 'spotify_artists', 'uri'])
        writer.writerows(matched)
    with open(NOT_FOUND_CSV, 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['trackname', 'artistname'])
        writer.writerows(missing)

    counts = ', '.join(f'{m}: {sum(row[0] == m for row in matched)}' for m in MATCH_ORDER)
    print(f'\nmatched {len(matched)}/{len(tracks)} ({len(uris)} unique), {len(missing)} not found')
    print(f'match quality: {counts}')
    print(f'length: {length_ranks.count(0)} agree, {length_ranks.count(2)} differ, {length_ranks.count(1)} unknown')
    print(f'review: {MATCHED_CSV} and {NOT_FOUND_CSV}')
    return uris

## Building the playlist

In [ ]:
def read_matched_uris(csv_path):
    """URIs from a hand-edited matched.csv, in the order of the original track list.
    Accepts spotify:track:... or open.spotify.com links."""
    position = {track: n for n, track in enumerate(read_tracks(csv_path))}
    with open(MATCHED_CSV, newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    # rows not in the track list (e.g. renamed by hand) go at the end
    rows.sort(key=lambda row: position.get((row['trackname'].strip(), row['artistname'].strip()), len(position)))

    uris = []
    for row in rows:
        value = row['uri'].strip()
        if not value:
            continue
        m = re.search(r'track[/:]([A-Za-z0-9]{22})', value)
        if m is None:
            print(f'skipping unrecognised uri for {row["trackname"]} - {row["artistname"]}: {value}')
            continue
        uri = f'spotify:track:{m[1]}'
        if uri not in uris:
            uris.append(uri)
    return uris


def create_playlist(sp, name, public):
    try:
        playlist = sp._post('me/playlists', payload={'name': name, 'public': public})
    except spotipy.SpotifyException as e:
        if e.http_status != 404:
            raise
        # older API shape
        playlist = sp.user_playlist_create(sp.me()['id'], name, public=public)
    if playlist is None:
        raise RuntimeError('Spotify returned no playlist data')
    return playlist


def add_items(sp, playlist_id, uris):
    # spotify takes at most 100 tracks per request
    for i in range(0, len(uris), 100):
        sp.playlist_add_items(playlist_id, uris[i:i + 100])

## Counting requests

In [ ]:
class CountingSpotify(spotipy.Spotify):
    """spotipy.Spotify that counts the API requests it sends, by endpoint."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.requests = Counter()

    def _internal_call(self, method, url, payload, params):
        # every request goes through here; 'search' / 'playlists/<id>/items' / 'me/playlists' -> first part
        self.requests[url.split('?')[0].replace(self.prefix, '').split('/')[0]] += 1
        return super()._internal_call(method, url, payload, params)


def format_requests(counts):
    """'6 (playlists: 5, me: 1)'"""
    detail = ', '.join(f'{name}: {n}' for name, n in counts.most_common())
    return f'{sum(counts.values())}' + (f' ({detail})' if detail else '')

---
# Run

## Log in

In [ ]:
load_dotenv(HERE / '.env')

# retries=0: fail fast on a rate limit instead of sleeping for Retry-After (can be hours)
sp = CountingSpotify(retries=0, auth_manager=SpotifyOAuth(
    redirect_uri=REDIRECT_URI, scope=SCOPE, cache_path=HERE / '.spotify_token_cache',
    # random value checked on the login redirect, so a forged redirect can't log you in
    state=secrets.token_urlsafe(16)))

## Option A: search and write the review files

Searches only tracks not already in `search_cache.json`, then writes to `matched.csv` and `not_found.csv`. Stop here to review, or go on to **Create the playlist**.

In [ ]:
before = sp.requests.copy()
try:
    tracks = read_tracks(CSV)
    durations = read_durations(CSV)
    if search_all(sp, tracks, durations):
        uris = write_reports(tracks, durations)
finally:
    print(f'Spotify requests: {format_requests(sp.requests - before)} (this cell)')

## Option B: use your edited matched.csv

If you skip Option A and run this after editing `matched.csv` manually [torture].

In [ ]:
before = sp.requests.copy()
try:
    uris = read_matched_uris(CSV)
    print(f'{len(uris)} unique tracks in {MATCHED_CSV.name}')
finally:
    print(f'Spotify requests: {format_requests(sp.requests - before)} (this cell)')

## Create the playlist

Makes a **new** playlist every time it runs, using `uris` from Option A or B.

In [ ]:
before = sp.requests.copy()
try:
    playlist = create_playlist(sp, PLAYLIST_NAME, PUBLIC)
    add_items(sp, playlist['id'], uris)
    print(f'created playlist: {playlist["external_urls"]["spotify"]}')
finally:
    print(f'Spotify requests: {format_requests(sp.requests - before)} (this cell)')